# Notebook 6: Phase 2 — Image training & late fusion

Train a **pretrained EfficientNet** (timm) with the **same patient-grouped folds** as Notebook 4, build **image OOF probabilities**, then **fuse** them with the tabular ensemble (`tabular_oof`) by searching blend weights on OOF (probability or logit mix — whichever maximizes **pAUC**).

**Prerequisites**
1. Complete **Notebook 3** (features) and **Notebook 4** (tabular models) so `outputs/model_predictions.pkl` exists.
2. Install image extras: `pip install -e ".[image]"` (PyTorch, timm, albumentations, h5py, Pillow).
3. Place competition images under `ISIC 2024 Skin Cancer Challenge Dataset/train-image/image/` with filenames `{isic_id}.jpg` (Kaggle layout).

**Outputs:** Updates `outputs/model_predictions.pkl` with `img_oof`, `final_oof` (fusion), `has_img=True`, and saves per-fold CNN weights under `outputs/models/`.

**Runtime:** Full 5-fold training on ~400k rows is **GPU-hours**; reduce `TrainingConfig.img_epochs`, `img_neg_subsample_fraction`, or `img_batch_size` for dry runs.


In [1]:
# ============================================================
# Setup — GPU recommended (CUDA or Apple MPS)
# ============================================================
import os, sys, pickle, warnings
from datetime import datetime

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/Skin-Cancer-Detection-ISIC-2024-"
else:
    ROOT = os.getcwd()

_SRC = os.path.abspath(os.path.join(ROOT, "src"))
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

from isic_challenge.config import TrainingConfig
from isic_challenge.metrics import compute_all_metrics

CFG = TrainingConfig()
_paths = CFG.paths(ROOT)
os.makedirs(_paths["model_dir"], exist_ok=True)

def ts():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"[{ts()}] ROOT={ROOT}")
print(
    f"[{ts()}] Phase 2 | backbone={CFG.img_model}, size={CFG.img_size}, "
    f"epochs={CFG.img_epochs}, neg_subsample={CFG.img_neg_subsample_fraction}"
)


[2026-04-19 16:40:26] ROOT=/Users/mahi0606/Desktop/Skin-Cancer-Detection-ISIC-2024-
[2026-04-19 16:40:26] Phase 2 | backbone=efficientnet_b0, size=224, epochs=10, neg_subsample=0.12


### Training visibility (best practice)

- **tqdm** bars: CV folds → epochs → train/val **batches** (rates and ETA).
- **Timestamped lines** (`[YYYY-MM-DD HH:MM:SS]`): fold summary, each epoch’s **train loss**, **val pAUC**, **LR**, early stop.
- **DataLoaders** use `persistent_workers` + `prefetch_factor` when `img_num_workers > 0`.
- Toggle noise: set `CFG.img_show_progress = False` (no tqdm) and/or `CFG.img_log_each_epoch = False` (keep tqdm, drop per-epoch log lines).


In [ ]:
# Optional: verbosity (defaults are True — full progress)
# CFG.img_show_progress = True   # tqdm: folds, epochs, batches
# CFG.img_log_each_epoch = True  # one timestamped line per epoch
# CFG.img_num_workers = 2        # faster loading on Linux/CUDA (macOS often 0)
pass


In [ ]:
# ============================================================
# Load Phase 1 predictions (tabular OOF + folds)
# ============================================================
pred_path = os.path.join(ROOT, "outputs", "model_predictions.pkl")
if not os.path.isfile(pred_path):
    raise FileNotFoundError(f"Missing {pred_path}. Run Notebook 4 first.")

with open(pred_path, "rb") as f:
    pred_data = pickle.load(f)

for k in ("y", "tabular_oof", "df_feat", "fold_iterator"):
    if k not in pred_data:
        raise KeyError(f"model_predictions.pkl missing key {k!r}")

print(f"Loaded pickle: phase={pred_data.get('phase')!r}, n={len(pred_data['y']):,}")


In [ ]:
# ============================================================
# Optional: quick image path sanity check
# ============================================================
from isic_challenge.image_pipeline import image_coverage_fraction

train_img_dir = _paths["train_img_dir"]
print(f"train_img_dir: {train_img_dir}")
if os.path.isdir(train_img_dir):
    frac = image_coverage_fraction(pred_data["df_feat"], train_img_dir)
    print(f"Approx. resolvable image files (sampled): {frac:.1%}")
else:
    print("WARNING: train_img_dir does not exist — download Kaggle train images into this folder.")


In [ ]:
# ============================================================
# Phase 2 — CNN OOF + late fusion (writes updated pickle)
# ============================================================
from isic_challenge.image_pipeline import run_phase2_image_oof_and_fusion

updated = run_phase2_image_oof_and_fusion(root=ROOT, cfg=CFG, pred_data=pred_data)

y = updated["y"]
tabular_oof = updated["tabular_oof"]
img_oof = updated["img_oof"]
final_oof = updated["final_oof"]

print("--- OOF metrics (full training rows) ---")
for name, p in [
    ("Tabular ensemble", tabular_oof),
    (CFG.img_model + " (image)", img_oof),
    ("Fused (final_oof)", final_oof),
]:
    m = compute_all_metrics(y, p)
    print(name, {k: round(v, 6) if isinstance(v, float) else v for k, v in m.items()})

print("Fusion:", updated.get("fusion_mode"), "w_tabular=", round(updated.get("fusion_weight_tabular", 0), 4))
print("Per-fold image model:", *updated.get("phase2_fold_metrics", []), sep="\n  ")


In [ ]:
# ============================================================
# Save updated predictions for Notebook 5
# ============================================================
out_path = os.path.join(ROOT, "outputs", "model_predictions.pkl")
with open(out_path, "wb") as f:
    pickle.dump(updated, f)
print(f"[{ts()}] Wrote {out_path} (phase={updated.get('phase')!r}, has_img={updated.get('has_img')})")
print("Next: run 05_Evaluation_and_Analysis.ipynb to compare curves and error analysis.")
